In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for dire in _:
        print(os.path.join(dirname, dire))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# !sudo apt install --allow-change-held-packages libcudnn8=8.1.0.77-1+cuda11.2 --yes
# !sudo pip install -q -U tensorflow-text tensorflow
# !sudo pip install PyArabic

In [4]:
from tqdm import tqdm 
import numpy as np
import tensorflow as tf 
from tensorflow import keras

import matplotlib.pyplot as plt
import unicodedata
import re

import os
from pyarabic.araby import tokenize, is_arabicrange, strip_tashkeel, normalize_hamza,\
                           strip_tatweel, strip_lastharaka
from pyarabic.trans import normalize_digits

In [5]:
def digit_convertor(text):
    return normalize_digits(text, source='all', out='east')
def tasheel_hamza(text):
    return normalize_hamza(text, method="tasheel")

def arabic_normlizer(text):
    text = tasheel_hamza(text)
    text = digit_convertor(text)
    text = strip_tashkeel(text)
    text = strip_tatweel(text)
    text = strip_lastharaka(text)
    return text

In [6]:
tt = '<start> أحمد محمد الأمة قاطبةُ سباعى 90 قائم . <end>'
arabic_normlizer(tt) 


'<start> احمد محمد الامة قاطبة سباعى ٩٠ قايم . <end>'

In [7]:
def unicode_to_ascii(s):
    '''
    Args:
        s : UniCode file
    Returns:
        ASCII file
    
    Converts the unicode file to ascii
    For each character:
                        there are two normal forms: normal form C and normal form D. 
                                                    - Normal form D (NFD) is also known as canonical decomposition,
                                                      and translates each character into its decomposed form. 
                                                      
                                                    - Normal form C (NFC) first applies a canonical decomposition, 
                                                    then composes pre-combined characters again.
    '''
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn') # Mn ==> Mark, Nonspacing (التشكيل )


def preprocess_sentence(s, i=0):
    '''
    Args:
        s : A single sentence 
    Returns:
        s : Single normalize sentence 
    
    Convert Unicode to ASCII
    Creating a space between a word and the punctuation following it
    eg: "he is a boy." => "he is a boy ." 
    Reference:- https://stackoverflow.com/questions/3645931/python-padding-punctuation-with-white-spaces-keeping-punctuation
    
    Replacing everything with space except (a-z, A-Z, ا-ي ".", "?", "!", ",")
    
    Adding a start and an end token to the sentence
    
    '''
    if i:
        s = arabic_normlizer(s)
        s = unicode_to_ascii(s.lower().strip())
    
        s = re.sub(r"([?.!,¿؟])", r" \1 ", s)
        s = re.sub(r'[" "]+', " ", s)
        s = re.sub(r"[^٠-٩؀-ۿ?.!,¿]+", " ", s)
        
    else:
        s = unicode_to_ascii(s.lower().strip())
    
        s = re.sub(r"([?.!,¿])", r" \1 ", s)
        s = re.sub(r'[" "]+', " ", s)
        s = re.sub(r"[^a-zA-Z0-9?.!,¿]+", " ", s)
    
    s = s.rstrip().strip()
    s = f'<start> {s} <end>'
    return s

In [8]:
class TokenHandler(object):
    def __init__(self, mapping):
        
        self.mapping = mapping
        self.mapping['UNK'] = 0
        self.reverse_mapping = dict([(i, v) for v, i in self.mapping.items()])
    
    def _get_ids(self, word):
        return self.mapping.get(word, 0)
    
    def _get_word(self, id):
        return self.reverse_mapping.get(id, 'UNK')
    
    def _unk_remove(self, word):
        return word if 0 != self._get_ids(word) else ''
    
    def _get_len(self, sent):
        return len(sent)
    
    def numberize_sent(self, sent):
        sent = sent.split()
    
        return list(map(self._get_ids, sent)) #[1, 2 , 5]
    
    def denumbrize_sent(self, ids_list):
        sent = list(map(self._get_word, ids_list))
        
        sent = list(map(self._unk_remove, sent))
        
        return ' '.join(sent) 
    
    
    def __len__(self):
        return len(self.mapping)
    
    def prepare_sent(self, sent, max_len=None):
        sent = self.numberize_sent(sent.numpy().decode())
        if max_len is None:
            max_len = max(map(self._get_len, ids)) 
        data = np.zeros(max_len)
        stop = len(sent) if len(sent) <= max_len else max_len 
            
        data[:stop] = sent[:stop-1] + [sent[-1]]
        return data
            
            
    def prepare_dataset(self, sents, max_len=None, target=False):
        
        ids = list(map(self.numberize_sent, sents)) 
        
        if max_len is None:
            
            max_len = max(map(self._get_len, ids)) 
        
#         data = np.zeros((len(ids), max_len))#, dtype=np.float32
        
        if target:
            max_len = max_len -1
            data = np.zeros((len(ids), max_len))
            label = np.zeros((len(ids), max_len))
            
            for i in tqdm(range(len(data))):
                
                d = ids[i]
                
                stop = len(d)-1 if len(d) <= max_len else max_len
                
                data[i, :stop] = ids[i][:stop] #+ [ids[i][-1]]    '<start> ahmed salah <end>' 
                
                label[i, :stop] = ids[i][1:stop] + [ids[i][-1]]


            return data, label
        
        
        data = np.zeros((len(ids), max_len), )
        for i in tqdm(range(len(data))):
            d = ids[i]
            
            stop = len(d) if len(d) <= max_len else max_len 
            
            data[i, :stop] = ids[i][:stop-1] + [ids[i][-1]]

        return data

In [9]:
with open('/kaggle/input/en-ar-open-suptitle-pairs/data/vocublary/en_dict.txt') as f:
    en_mapping = dict([(mp.split(':')[0].strip(), int(mp.split(':')[1].strip())) for mp in f.readlines()])
    
with open('/kaggle/input/en-ar-open-suptitle-pairs/data/vocublary/ar_dict.txt') as f:
    ar_mapping = dict([(mp.split(':')[0].strip(), int(mp.split(':')[1].strip())) for mp in f.readlines()])
en_manager = TokenHandler(en_mapping)
ar_manager = TokenHandler(ar_mapping)

del en_mapping, ar_mapping




def key_fun(file_name):
    return int(file_name[53:-4])
en_path = '/kaggle/input/en-ar-open-suptitle-pairs/data/en'
en_data = sorted([os.path.join(en_path, f) for f in os.listdir(en_path)], key=key_fun)

ar_path = '/kaggle/input/en-ar-open-suptitle-pairs/data/ar'
ar_data = sorted([os.path.join(ar_path, f) for f in os.listdir(ar_path)], key=key_fun)
en_data[:5]


['/kaggle/input/en-ar-open-suptitle-pairs/data/en/text_0.txt',
 '/kaggle/input/en-ar-open-suptitle-pairs/data/en/text_1.txt',
 '/kaggle/input/en-ar-open-suptitle-pairs/data/en/text_2.txt',
 '/kaggle/input/en-ar-open-suptitle-pairs/data/en/text_3.txt',
 '/kaggle/input/en-ar-open-suptitle-pairs/data/en/text_4.txt']

In [10]:
def preprocess_data(en, ar):
    in_ar = ar[:-1]
    out_ar = ar[1:]
    return (en, in_ar), out_ar
# def remove_fun(inp, out):
#     print(tf.reduce_sum(inp[0], 1) > 4 )
#     return sum(inp[0]) > 4 

In [ ]:
import functools

In [11]:
en_dataset = tf.data.TextLineDataset(en_data).map(lambda x: tf.py_function(en_manager.prepare_sent, [x, 20], tf.float64))
ar_dataset = tf.data.TextLineDataset(ar_data).map(lambda x: tf.py_function(ar_manager.prepare_sent, [x, 20], tf.float64))
dataset = tf.data.Dataset.zip((en_dataset, ar_dataset)).map(preprocess_data).batch(batch_size=128, drop_remainder=True)\
                                                       .prefetch(tf.data.AUTOTUNE).cache()
# del en_dataset, ar_dataset


2023-01-03 17:50:14.829623: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-03 17:50:14.933390: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-03 17:50:14.934141: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:937] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-03 17:50:14.937628: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compil

In [12]:
for (i, j), l in dataset.as_numpy_iterator():
    print(i.shape)
    print(j.shape)
    print(l.shape)
    break

2023-01-03 17:50:17.615883: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:185] None of the MLIR Optimization Passes are enabled (registered 2)


(128, 20)
(128, 19)
(128, 19)


2023-01-03 17:50:17.872553: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [13]:
# val_data = dataset.take(4)

# dataset = dataset.skip(5)

# for (i, j), l in val_data.take(1):
#     print(i)
#     print()
#     print(j)
#     print()
#     print(l)
#     print()
#     print()
    
for (i, j), l in dataset.take(1):
    print(i)
    print()
    print(j)
    print()
    print(l)
    print()

tf.Tensor(
[[  0.  12.   7. ...   0.   0.   0.]
 [ 11.  19.  17. ...   0.   0.   0.]
 [ 11.  25.  21. ...  20.  12.   9.]
 ...
 [ 11.  19. 247. ...   0.   0.   0.]
 [ 11.  12. 581. ...   0.   0.   0.]
 [ 11. 181. 585. ...   0.   0.   0.]], shape=(128, 20), dtype=float64)

tf.Tensor(
[[  0.   8.   4. ...   0.   0.   0.]
 [ 10.  12.  14. ...   0.   0.   0.]
 [ 10.  25.  19. ...   0.   0.   0.]
 ...
 [ 10. 735. 297. ...   0.   0.   0.]
 [ 10. 737. 736. ...   0.   0.   0.]
 [ 10. 722. 726. ...   0.   0.   0.]], shape=(128, 19), dtype=float64)

tf.Tensor(
[[  8.   4.   9. ...   0.   0.   0.]
 [ 12.  14.  15. ...   0.   0.   0.]
 [ 25.  19.  20. ...   0.   0.   0.]
 ...
 [735. 297. 471. ...   0.   0.   0.]
 [737. 736. 739. ...   0.   0.   0.]
 [722. 726. 741. ...   0.   0.   0.]], shape=(128, 19), dtype=float64)



2023-01-03 17:50:18.075022: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


In [14]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, depth, mask_zero=False):
        super(PositionalEmbedding, self).__init__()

        self.depth = depth
        
        self.embedding = tf.keras.layers.Embedding(vocab_size, depth, mask_zero=mask_zero) 
        
        self.pos_encoding = self._get_positional_encoding(length=2048, depth=depth)

    def call(self, x):
        
        length = x.shape[1]   #[batch_size, timestep]
        
        x = self.embedding(x) #[batch_size, timestep, depth]
        # This factor sets the relative scale of the embedding and positonal_encoding.
        x *= tf.math.sqrt(tf.cast(self.depth, tf.float32))
        
        x = x + self.pos_encoding[tf.newaxis, :length, :] 

        return x
    
    def compute_mask(self, *args, **kwargs):
        return self.embedding.compute_mask(*args, **kwargs)

    
    def _get_positional_encoding(self, length, depth, n=10000):
        depth = depth

        positions = np.arange(length)[:, np.newaxis]     # (seq, 1)  [0, 1, 2, 3 ... length-1]

        depths = np.arange(depth)[np.newaxis, :]/depth   # (1, depth) [0 / depth, 1 / depth, 2/depth, 3/depth ... length-1/depth]
        
        angle_rates = 1 / (n**depths)         # (1, depth)

        angle_rads = positions * angle_rates      # (pos, depth)

        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])

        # apply cos to odd indices in the array; 2i+1
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])
#         print(angle_rads.shape)
        return tf.cast(angle_rads, dtype=tf.float32)

In [15]:
class BaseAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(BaseAttention, self).__init__()
        self.mha = tf.keras.layers.MultiHeadAttention(**kwargs)
        self.layernorm = tf.keras.layers.LayerNormalization()
        self.add = tf.keras.layers.Add()

In [16]:
class EncoderDecoder(BaseAttention):
    def call(self, x, context):
        attn_output, attn_scores = self.mha(
                                            query= x,
                                            key= context,
                                            value= context,
                                            return_attention_scores=True)

        # Cache the attention scores for plotting later.
        self.last_attn_scores = attn_scores

        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x

In [17]:
class SelfAttention(BaseAttention):
    def call(self, x):
        attn_output = self.mha(
                                query=x,
                                value=x,
                                key=x)
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x

In [18]:
class MaskedSelfAttention(BaseAttention):
    def call(self, x):
#         try:
            
        attn_output = self.mha(
                                query=x,
                                value=x,
                                key=x,
                                use_causal_mask = True)#attention_mask=mask
#         except Exception as e:
#                 print(e)
        x = self.add([x, attn_output])
        x = self.layernorm(x)
        return x

In [19]:
class FeedForward(tf.keras.layers.Layer):
    def __init__(self, d_model, dff, dropout_rate=0.1):
        super(FeedForward, self).__init__()
        self.seq = tf.keras.Sequential([
                                          tf.keras.layers.Dense(dff, activation='relu'),
                                          tf.keras.layers.Dense(d_model, ),
                                          tf.keras.layers.Dropout(dropout_rate)])
        self.add = tf.keras.layers.Add()
        self.layer_norm = tf.keras.layers.LayerNormalization()

    def call(self, x):
        x = self.add([x, self.seq(x)])
        x = self.layer_norm(x) 
        return x

In [20]:
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self,*, d_model, num_heads, dff, dropout_rate=0.1):
        super().__init__()

        self.self_attention = SelfAttention(
                                            num_heads=num_heads,
                                            key_dim=d_model,
                                            dropout=dropout_rate)

        self.ffn = FeedForward(d_model, dff)

    def call(self, x):
        x = self.self_attention(x)
        x = self.ffn(x)
        return x

In [21]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads, dff, vocab_size, dropout_rate=0.1):
        super(Encoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.pos_embedding = PositionalEmbedding(vocab_size=vocab_size, depth=d_model, mask_zero=True)

        self.enc_layers = [EncoderLayer(d_model=d_model, num_heads=num_heads, dff=dff, dropout_rate=dropout_rate)
                            for _ in range(num_layers)]
        
        self.dropout = tf.keras.layers.Dropout(dropout_rate)

    def call(self, x):
        # `x` is token-IDs shape: (batch, seq_len)
        x = self.pos_embedding(x)  # Shape `(batch_size, seq_len, d_model)`.

        # Add dropout.
        x = self.dropout(x)

        for i in range(self.num_layers):
            x = self.enc_layers[i](x)

        return x  

In [22]:
class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, *, d_model, num_heads, dff, dropout_rate=0.1):
        super(DecoderLayer, self).__init__()

        self.causal_self_attention = MaskedSelfAttention(
                                                        num_heads=num_heads,
                                                        key_dim=d_model,
                                                        dropout=dropout_rate)

        self.cross_attention = EncoderDecoder(
                                            num_heads=num_heads,
                                            key_dim=d_model,
                                            dropout=dropout_rate)

        self.ffn = FeedForward(d_model, dff)

    def call(self, x, context):
        
        x = self.causal_self_attention(x=x)#, mask=mask
        
        x = self.cross_attention(x=x, context=context)

        # Cache the last attention scores for plotting later
        self.last_attn_scores = self.cross_attention.last_attn_scores

        x = self.ffn(x)  # Shape `(batch_size, seq_len, d_model)`.
        return x

In [23]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, *, num_layers, d_model, num_heads, dff, vocab_size, dropout_rate=0.1):
        super(Decoder, self).__init__()

        self.d_model = d_model
        self.num_layers = num_layers

        self.pos_embedding = PositionalEmbedding(vocab_size=vocab_size, depth=d_model, mask_zero=True)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        
        self.dec_layers = [DecoderLayer(d_model=d_model, num_heads=num_heads, dff=dff, dropout_rate=dropout_rate)
                                        for _ in range(num_layers)]

        self.last_attn_scores = None

    def call(self, x, context):
        z = x
        # `x` is token-IDs shape (batch, target_seq_len)
        x = self.pos_embedding(x)  # (batch_size, target_seq_len, d_model)

        x = self.dropout(x)

        for i in range(self.num_layers):
            x  = self.dec_layers[i](x, context)#
            

        self.last_attn_scores = self.dec_layers[-1].last_attn_scores

        # The shape of x is (batch_size, target_seq_len, d_model).
        return x

In [24]:
class Transformer(tf.keras.Model):
    def __init__(self, *, num_layers, d_model, num_heads, dff,
               input_vocab_size, target_vocab_size, dropout_rate=0.1):
        super().__init__()
        self.encoder = Encoder(num_layers=num_layers, d_model=d_model,
                               num_heads=num_heads, dff=dff,
                               vocab_size=input_vocab_size,
                               dropout_rate=dropout_rate)

        self.decoder = Decoder(num_layers=num_layers, d_model=d_model,
                               num_heads=num_heads, dff=dff,
                               vocab_size=target_vocab_size,
                               dropout_rate=dropout_rate)

        self.final_layer = tf.keras.layers.Dense(target_vocab_size)

    def call(self, inputs):
        # To use a Keras model with `.fit` you must pass all your inputs in the
        # first argument.
        context, x  = inputs

        context = self.encoder(context)  # (batch_size, context_len, d_model)

        x = self.decoder(x, context)  # (batch_size, target_len, d_model)

        # Final linear layer output.
        logits = self.final_layer(x)  # (batch_size, target_len, target_vocab_size)

        try:
          # Drop the keras mask, so it doesn't scale the losses/metrics.
          # b/250038731
            del logits._keras_mask
        except AttributeError:
            pass

        # Return the final output and the attention weights.
        return logits

In [25]:
num_layers = 4
d_model = 128
dff = 512
num_heads = 8
dropout_rate = 0.1

transformer = Transformer(
                            num_layers=num_layers,
                            d_model=d_model,
                            num_heads=num_heads,
                            dff=dff,
                            input_vocab_size=len(en_manager),
                            target_vocab_size=len(ar_manager),
                            dropout_rate=dropout_rate)

In [26]:
for (i, j), l in dataset.take(1):
    x = tf.argmax(transformer((i, j)), 2)[0]
    print(en_manager.denumbrize_sent(i[0].numpy()))
    print(ar_manager.denumbrize_sent(x.numpy()))

2023-01-03 17:50:22.078479: I tensorflow/stream_executor/cuda/cuda_dnn.cc:369] Loaded cuDNN version 8100
2023-01-03 17:50:22.723222: W tensorflow/core/kernels/data/cache_dataset_ops.cc:768] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


TypeError: call() got an unexpected keyword argument 'use_causal_mask'